In [5]:
# Run this cell to install and upgrade the necessary libraries
!pip install --upgrade pip
!pip install ragas langchain-openai datasets pandas nest_asyncio

In [6]:
import os
import pandas as pd
import asyncio
import nest_asyncio
from datasets import Dataset

# Required to run nested event loops safely within Jupyter Notebooks
nest_asyncio.apply()

from ragas import SingleTurnSample, EvaluationDataset
from ragas.metrics import LLMContextRecall, Faithfulness, ResponseRelevancy
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI

# FIX: Added the missing 's' at the very beginning ('sk-proj-...')
MY_API_KEY = "<Enter your APIs>"

# Update environment memory
os.environ["OPENAI_API_KEY"] = MY_API_KEY

print("\n=== Initializing the LLM-As-A-Judge Engines ===")

# Pass the corrected key to the model
openai_llm = ChatOpenAI(
    model="gpt-4o-mini", 
    temperature=0.0,
    openai_api_key=MY_API_KEY, 
    api_key=MY_API_KEY         
)

evaluator_llm = LangchainLLMWrapper(openai_llm)
print("Evaluator engine ready successfully!")


=== Initializing the LLM-As-A-Judge Engines ===
Evaluator engine ready successfully!


In [7]:
print("=== Preparing Synthetic Evaluation Samples ===")

# Creating mock logs from a production RAG system
samples = [
    SingleTurnSample(
        user_input="What color is the sky during a clear midday?",
        retrieved_contexts=[
            "The atmosphere scatters shorter wavelengths of light. This phenomenon, Rayleigh scattering, causes the midday sky to appear blue to an observer."
        ],
        response="The sky appears blue during midday due to Rayleigh scattering.",
        reference="The sky is blue during a clear day."
    ),
    SingleTurnSample(
        user_input="What is the capital city of France?",
        retrieved_contexts=[
            "Paris is a global hub for art and fashion. It is also the populous capital city of France situated on the Seine River."
        ],
        # Simulated Hallucination: The context says Paris, but the model generated London!
        response="The capital city of France is London.", 
        reference="The capital city of France is Paris."
    )
]

# Convert standard list elements into an official Ragas Evaluation Dataset
eval_dataset = EvaluationDataset(samples=samples)

print("=== Binding Evaluator Metrics ===")
# Link our metrics to the OpenAI judge engine we configured in Cell 2
faithfulness_metric = Faithfulness(llm=evaluator_llm)
recall_metric = LLMContextRecall(llm=evaluator_llm)
relevancy_metric = ResponseRelevancy(llm=evaluator_llm)

print("Metrics successfully linked and ready for execution.")

=== Preparing Synthetic Evaluation Samples ===
=== Binding Evaluator Metrics ===
Metrics successfully linked and ready for execution.


In [8]:
from langchain.embeddings import OpenAIEmbeddings

async def evaluate_samples():
    scores_record = []
    
    # Loop through the data to compute individual scores
    for i, sample in enumerate(samples):
        print(f"Scoring Sample {i+1}...")
        
        # Compute individual metrics asynchronously using the judge LLM
        faith_score = await faithfulness_metric.single_turn_ascore(sample)
        recall_score = await recall_metric.single_turn_ascore(sample)
        rel_score = await relevancy_metric.single_turn_ascore(sample)
        
        scores_record.append({
            "Query": sample.user_input,
            "Response": sample.response,
            "Faithfulness (No Hallucination)": faith_score,
            "Context Recall (Retriever Quality)": recall_score,
            "Response Relevancy (Alignment)": rel_score
        })
    return scores_record

# Run the asynchronous evaluation function
embeddings = OpenAIEmbeddings(openai_api_key=MY_API_KEY)
relevancy_metric = ResponseRelevancy(llm=evaluator_llm, embeddings=embeddings)

results = asyncio.run(evaluate_samples())

print("\n=== FINAL RAGAS METRIC EVALUATION ===")
# Convert results into a Pandas DataFrame for instant, clear tabular display
df = pd.DataFrame(results)
df

Scoring Sample 1...
Scoring Sample 2...

=== FINAL RAGAS METRIC EVALUATION ===


,Query,Response,Faithfulness (No Hallucination),Context Recall (Retriever Quality),Response Relevancy (Alignment)
0,What color is the sky during a clear midday?,The sky appears blue during midday due to Rayl...,1.0,1.0,0.919516
1,What is the capital city of France?,The capital city of France is London.,0.0,1.0,1.000000
